In [1]:
!git clone https://github.com/facebookresearch/segment-anything-2
%cd /kaggle/working/segment-anything-2
!pip install -q -e .

Cloning into 'segment-anything-2'...
remote: Enumerating objects: 1057, done.
remote: Counting objects: 100% (426/426), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1057 (delta 273), reused 248 (delta 248), pack-reused 631 (from 2)
Receiving objects: 100% (1057/1057), 121.74 MiB | 20.47 MiB/s, done.
Resolving deltas: 100% (384/384), done.
/kaggle/working/segment-anything-2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 7.5 MB/s eta 0:00:000:00:010:01
  Building editable for SAM-2 (pyproject.toml) ... done
ERROR: pip's dependency resolver does not curr

In [2]:
!wget -O sam2_hiera_base_plus.pt "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_base_plus.pt"
%cd /kaggle/working/segment-anything-2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

--2024-12-24 07:43:32--  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_base_plus.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.7.38, 13.35.7.82, 13.35.7.50, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.7.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 323493298 (309M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2_hiera_base_plus.pt’

sam2_hiera_base_plu 100%[===================>] 308.51M  96.1MB/s    in 3.4s    

2024-12-24 07:43:35 (92.0 MB/s) - ‘sam2_hiera_base_plus.pt’ saved [323493298/323493298]

/kaggle/working/segment-anything-2


In [24]:
yaml_content="""
# @package _global_

# Model
model:
  _target_: sam2.modeling.sam2_base.SAM2Base
  image_encoder:
    _target_: sam2.modeling.backbones.image_encoder.ImageEncoder
    scalp: 1
    trunk:
      _target_: sam2.modeling.backbones.hieradet.Hiera
      embed_dim: 112
      num_heads: 2
    neck:
      _target_: sam2.modeling.backbones.image_encoder.FpnNeck
      position_encoding:
        _target_: sam2.modeling.position_encoding.PositionEmbeddingSine
        num_pos_feats: 256
        normalize: true
        scale: null
        temperature: 10000
      d_model: 256
      backbone_channel_list: [896, 448, 224, 112]
      fpn_top_down_levels: [2, 3]  # output level 0 and 1 directly use the backbone features
      fpn_interp_model: nearest

  memory_attention:
    _target_: sam2.modeling.memory_attention.MemoryAttention
    d_model: 256
    pos_enc_at_input: true
    layer:
      _target_: sam2.modeling.memory_attention.MemoryAttentionLayer
      activation: relu
      dim_feedforward: 2048
      dropout: 0.1
      pos_enc_at_attn: false
      self_attention:
        _target_: sam2.modeling.sam.transformer.RoPEAttention
        rope_theta: 10000.0
        feat_sizes: [64, 64]
        embedding_dim: 256
        num_heads: 1
        downsample_rate: 1
        dropout: 0.1
      d_model: 256
      pos_enc_at_cross_attn_keys: true
      pos_enc_at_cross_attn_queries: false
      cross_attention:
        _target_: sam2.modeling.sam.transformer.RoPEAttention
        rope_theta: 10000.0
        feat_sizes: [64, 64]
        rope_k_repeat: True
        embedding_dim: 256
        num_heads: 1
        downsample_rate: 1
        dropout: 0.1
        kv_in_dim: 64
    num_layers: 4

  memory_encoder:
      _target_: sam2.modeling.memory_encoder.MemoryEncoder
      out_dim: 64
      position_encoding:
        _target_: sam2.modeling.position_encoding.PositionEmbeddingSine
        num_pos_feats: 64
        normalize: true
        scale: null
        temperature: 10000
      mask_downsampler:
        _target_: sam2.modeling.memory_encoder.MaskDownSampler
        kernel_size: 3
        stride: 2
        padding: 1
      fuser:
        _target_: sam2.modeling.memory_encoder.Fuser
        layer:
          _target_: sam2.modeling.memory_encoder.CXBlock
          dim: 256
          kernel_size: 7
          padding: 3
          layer_scale_init_value: 1e-6
          use_dwconv: True  # depth-wise convs
        num_layers: 2

  num_maskmem: 7
  image_size: 1024
  # apply scaled sigmoid on mask logits for memory encoder, and directly feed input mask as output mask
  sigmoid_scale_for_mem_enc: 20.0
  sigmoid_bias_for_mem_enc: -10.0
  use_mask_input_as_output_without_sam: true
  # Memory
  directly_add_no_mem_embed: true
  # use high-resolution feature map in the SAM mask decoder
  use_high_res_features_in_sam: true
  # output 3 masks on the first click on initial conditioning frames
  multimask_output_in_sam: true
  # SAM heads
  iou_prediction_use_sigmoid: True
  # cross-attend to object pointers from other frames (based on SAM output tokens) in the encoder
  use_obj_ptrs_in_encoder: true
  add_tpos_enc_to_obj_ptrs: false
  only_obj_ptrs_in_the_past_for_eval: true
  # object occlusion prediction
  pred_obj_scores: true
  pred_obj_scores_mlp: true
  fixed_no_obj_ptr: true
  # multimask tracking settings
  multimask_output_for_tracking: true
  use_multimask_token_for_obj_ptr: true
  multimask_min_pt_num: 0
  multimask_max_pt_num: 1
  use_mlp_for_obj_ptr_proj: true
  # Compilation flag
  compile_image_encoder: False
  
"""
! mkdir /kaggle/working/sam2_hiera_b+.yaml
yaml_file_path = "/kaggle/working/segment-anything-2/sam2_hiera_b+.yaml"
with open(yaml_file_path, "w") as f:
    f.write(yaml_content)

# from hydra import initialize_config_dir, compose
# from hydra.core.global_hydra import GlobalHydra
# import os

# # Clear previous Hydra runs
# if GlobalHydra().is_initialized():
#     GlobalHydra().clear()

# # Initialize Hydra with the directory containing the YAML file
# config_dir = "/kaggle/working"
# initialize_config_dir(config_dir=config_dir)

# # Load the configuration using Hydra
# config_name = "sam2_hiera_b+"
# model_cfg = compose(config_name=config_name)

mkdir: cannot create directory ‘/kaggle/working/sam2_hiera_b+.yaml’: File exists


In [3]:
sam2_checkpoint = "/kaggle/working/segment-anything-2/sam2_hiera_base_plus.pt"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)

# Load fine-tuned weights if available
SAM_MODEL_PATH = "/kaggle/input/cell_sam/pytorch/v1/1/fine_tuned_sam2.torch"
predictor.model.load_state_dict(torch.load(SAM_MODEL_PATH))

NameError: name 'model_cfg' is not defined

In [7]:
import torch
import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from segment_anything import SamPredictor, sam_model_registry
from pycocotools.mask import toBbox

import hydra
from sam2.build_sam import build_sam2

# hydra is initialized on import of sam2, which sets the search path which can't be modified
# so we need to clear the hydra instance
hydra.core.global_hydra.GlobalHydra.instance().clear()
# reinit hydra with a new search path for configs
hydra.initialize_config_module('/kaggle/working/segment-anything-2/sam2/configs/sam2', version_base='1.2')


SAM_MODEL_PATH = "/kaggle/input/cell_sam/pytorch/v1/1/fine_tuned_sam2.torch"
sam2_checkpoint = "/kaggle/working/segment-anything-2/sam2_hiera_base_plus.pt"  # @param ["sam2_hiera_tiny.pt", "sam2_hiera_small.pt", "sam2_hiera_base_plus.pt", "sam2_hiera_large.pt"]
model_cfg = "/kaggle/working/segment-anything-2/sam2_hiera_b+.yaml" # @param ["sam2_hiera_t.yaml", "sam2_hiera_s.yaml", "sam2_hiera_b+.yaml", "sam2_hiera_l.yaml"]

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(SAM_MODEL_PATH))


MissingConfigException: Primary config module '.kaggle.working.segment-anything-2.sam2.configs.sam2' not found.
Check that it's correct and contains an __init__.py file

In [2]:
!mkdir /tmp/output
import shutil

mkdir: cannot create directory ‘/tmp/output’: File exists


In [3]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
INPUT_IMAGES_DIR = "/kaggle/input/nmcd-400/train"
INPUT_COCO_FILE = "/kaggle/input/nmcd-400/train/_annotations.coco.json" 
OUTPUT_MASKS_DIR = "/tmp/output/masks"
OUTPUT_JSON_FILE = "/kaggle/working/output.json"

os.makedirs(OUTPUT_MASKS_DIR, exist_ok=True)
# Load COCO Annotations
with open(INPUT_COCO_FILE, "r") as f:
    coco_data = json.load(f)

images_info = {img["id"]: img for img in coco_data["images"]}
annotations = coco_data["annotations"]


In [ ]:
def visualize_bounding_boxes(image_path, bboxes):
    """Visualize bounding boxes on an image."""
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    for bbox in bboxes:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
def calculate_iou(mask, bbox):
    """Calculate IoU between a mask and a bounding box."""
    x, y, w, h = bbox
    bbox_mask = np.zeros_like(mask, dtype=np.uint8)
    bbox_mask[int(y):int(y + h), int(x):int(x + w)] = 1
    intersection = np.logical_and(mask, bbox_mask).sum()
    union = np.logical_or(mask, bbox_mask).sum()
    return intersection / union if union > 0 else 0

def clip_mask_to_bbox(mask, bbox):
    """Clip the mask to fit within the bounding box."""
    x, y, w, h = map(int, bbox)
    clipped_mask = np.zeros_like(mask, dtype=np.uint8)
    clipped_mask[y:y + h, x:x + w] = mask[y:y + h, x:x + w]
    return clipped_mask

In [ ]:
import random
import matplotlib.pyplot as plt
random_image_id = random.choice(list(images_info.keys()))
random_image_info = images_info[random_image_id]
random_image_path = os.path.join(INPUT_IMAGES_DIR, random_image_info["file_name"])
random_bboxes = [ann["bbox"] for ann in annotations if ann["image_id"] == random_image_id]
visualize_bounding_boxes(random_image_path, random_bboxes)

In [ ]:
c=0
for ann in annotations:
    bbox=ann["bbox"]
    print(bbox)
    c+=1.
    if c>0:
        break

In [ ]:
output_data = {}
iou_scores = []
cell_id_counters = {}

if os.path.isfile(OUTPUT_MASKS_DIR):
    os.remove(OUTPUT_MASKS_DIR)
else:
    print("Error: %s file not found" % OUTPUT_MASKS_DIR)
os.makedirs(OUTPUT_MASKS_DIR, exist_ok=True)

for annotation in tqdm(annotations, desc="Processing annotations", dynamic_ncols=True):
    img_id = annotation["image_id"]
    bbox = annotation["bbox"]
    x,y,w,h=bbox
    x,y,w,h=x,y,x+w, y+h
    bbox_new=x,y,w,h
    # Load the image
    image_info = images_info[img_id]
    image_path = os.path.join(INPUT_IMAGES_DIR, image_info["file_name"])

    # Initialize entry if not exists
    if img_id not in output_data:
        output_data[img_id] = {
            "image_id": img_id,
            "image_path": image_path,
            "cell_details": []
        }
        cell_id_counters[img_id] = 0  # Initialize the cell counter for this image

    # Set SAM predictor
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)

    # Generate masks with bounding box as prompt
    masks, _, _ = predictor.predict(box=np.array(bbox_new))

    best_iou = 0
    best_mask = None

    for mask in masks:
        iou = calculate_iou(mask, bbox)
        if iou > best_iou:
            best_iou = iou
            best_mask = mask
        iou_scores.append(best_iou)
        
    if best_mask is None:
        # No mask found, use bounding box as mask
        best_mask = np.zeros_like(image[:, :, 0], dtype=np.uint8)
        x, y, w, h = map(int, bbox)
        best_mask[y:y + h, x:x + w] = 1

    # Clip the mask to the bounding box
    final_mask = clip_mask_to_bbox(best_mask, bbox)

    # Save the mask as an image
    cell_id_counters[img_id] += 1
    unique_cell_id = f"{img_id}_{cell_id_counters[img_id]}"
    mask_file_name = f"{unique_cell_id}.png"
    mask_path = os.path.join(OUTPUT_MASKS_DIR, mask_file_name)
    cv2.imwrite(mask_path, final_mask * 255)

    # Append cell details to the image entry
    output_data[img_id]["cell_details"].append({
        "id": unique_cell_id,
        "mask_path": mask_path,
        "label": annotation["category_id"],  # Assuming category_id corresponds to the label
        "bbox": annotation['bbox']
    })

# Convert output_data to a list
output_data_list = list(output_data.values())

# Save results to JSON
with open(OUTPUT_JSON_FILE, "w") as f:
    json.dump(output_data_list, f, indent=4)


# Save results to JSON
with open(OUTPUT_JSON_FILE, "w") as f:
    json.dump(output_data_list, f, indent=4)

average_iou = sum(iou_scores) / len(iou_scores)
print(f"Average IoU: {average_iou:.4f}")


In [ ]:
shutil.make_archive('/kaggle/working/output_masks', 'zip', OUTPUT_MASKS_DIR)

In [ ]:
file_path = "/kaggle/working/output_masks.zip"  # Replace with your file path
file_name = 'output_masks.zip'        # Desired name in Google Drive

# Create a Google Drive file instance
gdrive_file = drive.CreateFile({'title': file_name})
gdrive_file.SetContentFile(file_path)
gdrive_file.Upload()

print(f"File '{file_name}' uploaded to Google Drive.")

In [ ]:
with open(OUTPUT_JSON_FILE) as f:
    output_data=json.load(f)

for data in output_data:
    image_path=data["image_path"]
    mask_path=data["cell_details"][0]["mask_path"]
    image = cv2.imread(image_path)
    image_rgb= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mask=cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    print(image.shape)
    mask_3d=np.dstack([mask]*3)
    #mask_3d = np.dstack([mask_3d, mask])
    mask_3d = mask_3d.astype(np.uint8)
    print(mask_3d.shape)
    image_with_mask = cv2.addWeighted(image_rgb, 1 - 0.5, mask_3d, 0.5, 0)
    print(image_with_mask.shape)
    plt.imshow(image_with_mask)
    break

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json


def overlay_transparent_masks_from_json(json_file, alpha=0.5):
    """Overlay transparent masks on images from a saved JSON file."""
    
    # Load the JSON data
    with open(json_file, 'r') as f:
        output_data = json.load(f)
    
    for data in output_data:
        image_path = data["image_path"]
        mask_path = data["mask_path"]
        
        # Read the image and the mask
        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  # Load mask as grayscale

        # Resize the mask to match the image size
        mask_resized = cv2.resize(mask, (image.shape[1], image.shape[0]))

        # Create a 3-channel version of the mask for overlaying
        mask_3d = np.dstack([mask_resized] * 3)  # Convert mask to 3 channels (RGB)
        
        # Convert mask to RGBA (add transparency channel)
        #mask_3d = np.dstack([mask_3d, mask_resized])  # Add the alpha channel to the mask

        # Make sure the mask has transparency (alpha channel)
        mask_3d = mask_3d.astype(np.uint8)

        # Overlay the transparent mask on the image using transparency (alpha blending)
        image_with_mask = cv2.addWeighted(image_rgb, 1 - alpha, mask_3d, alpha, 0)

        # Display the final image with transparent mask overlay
        plt.figure(figsize=(10, 10))
        plt.imshow(image_with_mask)
        plt.axis('off')
        plt.show()

overlay_transparent_masks_from_json(OUTPUT_JSON_FILE, alpha=0.25)

